#### setup

In [ ]:
# %% 0 - Setup:
if __name__ == "__main__":
    import sys, pathlib
    sys.path.insert(0, str(pathlib.Path.cwd() / "src"))

    %config InlineBackend.figure_format = "retina"

    from meanexit import (TOY, rng_factory, u_exact, grid_sweep, figures)

    MASTER_SEED = None            # None -> fresh randomness on every re-run
    get_rng = rng_factory(MASTER_SEED)

    print("toy problem    :", TOY)
    print(f"   exact u(X0) = {float(u_exact(TOY.X0, TOY)):.4f}          (12.4)")

# Mean Exit Times &nbsp;&nbsp;

*An Introduction to the Numerical Simulation of Stochastic Differential Equations*

<span style="font-variant: small-caps;">D. J. Higham</span> and <span style="font-variant: small-caps;">P. E. Kloeden</span>, 

---

## Outline:
1. **Motivation & Problem Statement**
2. **Monte Carlo - Euler Maruyama**
3. **Analytical Baseline to Mean Exit Times**
4. **Numerical Example Problem**
5. **Outro**


**Presenter:** *A. Cevher Uysal*  
**Date:** *16 July 2026*

---

## §1 · Motivation & Problem Definition &nbsp;&nbsp;

$$\;\mathrm{d}X(t) = f(X(t))\,\mathrm{d}t + g(X(t))\, \mathrm{d}W(t), \qquad X(0)=X_0\in(a,b)  \tag{12.1}$$

$$\;T_{\mathrm{exit}} := \inf\{t : X(t)=a \text{ or } X(t)=b\}$$

$$\textbf{Goal:}\text{ estimate } \; T^{\mathrm{mean}}_{\mathrm{exit}} = \mathbb{E}[T_{\mathrm{exit}}]$$

---

## §2 · Monte Carlo — Euler–Maruyama &nbsp;&nbsp;

Simulate $M$ sample paths with the EM method,
stop each at the first *grid point* outside $(a,b)$, average sample exit times.

```text
 1   choose a stepsize Δt
 2   choose a number of paths M
 3   for s = 1 to M
 4       set t_n = 0 and X_n = X0
 5       while a < X_n < b
 6           compute a N(0,1) sample ξ_n
 7           replace X_n by X_n + Δt f(X_n) + √Δt ξ_n g(X_n)             ← (EM step)
 8           replace t_n by t_n + Δt
 9       end
10       set T_exit^s = t_n − ½ Δt                       (midpoint of the last step)
11   end
12   set a_M  = (1/M) Σ_s T_exit^s
13   set b_M² = (1/(M−1)) Σ_s (T_exit^s − a_M)²
```

$$\text{95\% confidence interval:}  \qquad
\Big[\,a_M - 1.96\,\tfrac{b_M}{\sqrt{M}},\;\; a_M + 1.96\,\tfrac{b_M}{\sqrt{M}}\,\Big]$$

---

### §2.1 · Sources of Error for MC-EM &nbsp;&nbsp;

Two sources of error *we have already covered* in previous chapters

1. **Sampling error** — a sample mean is not an expectation. 
2. **Discretization error** — an EM path is not an SDE path.

In [ ]:
# Error source 1 - MC Sampling

# Experiment: histogram of the m=1000 EM-simulated exit times 
#               and the running sample mean with its shrinking 95% CI
#               for M=10^5 paths, Δt = 10^-1.

fig, res = figures.fig_sampling_error(TOY, get_rng("sampling"))

In [ ]:
# Error source 2 - Discretization

# Experiment: Exact solution and EM simulation differences for two stepsizes
#               for T=32, Δt ∈ { 1, 0.25 }.

fig = figures.fig_discretization_error(TOY, get_rng("discretization"))

---

### §2.1 · Sources of Error for MC-EM (cont'd) &nbsp;&nbsp;

And a new source of error inherent to *discrete sampling*

3. **Missed exits error** — we record solution values **only at the grid points** $\{t_i\}$: 
   

within $t_i < t < t_{i+1}$
the path may leave $(a,b)$ *and return unnoticed*

In [ ]:
# Error source 3 - Missed Exits

# Experiment: Plot exact solution to find instances with missed exits
#               for Δt = 0.618.

fig = figures.fig_missed_exit(TOY, get_rng("missed-exits"))

---

## §3 · Analytical Baseline &nbsp;&nbsp;

$$u(x) := \mathbb{E}[T_{exit} \mid X(0) = x]$$

In [ ]:
# Bias to Exact

# Experiment: Plot exact solution to find instances with missed exits
#               for Δt = 0.618.

fig = figures.fig_bias_to_exact(TOY, get_rng("bias-to-exact"))

---

## §4 · A Toy Problem &nbsp;&nbsp;

Consider *Geometric Brownian Motion (GBM)*, where we substitute $f(X) = \mu \cdot X(t)$ and $g(X) = \sigma \cdot X(t)$ into (12.1) to yield:

$$\mathrm{d}X(t) = \mu X(t)\,\mathrm{d}t + \sigma X(t)\,\mathrm{d}W(t) \tag{5.4}$$

with **exact solution** and **mean**

$$X(t) = X_0\,e^{(\mu-\frac{1}{2}\sigma^2)t + \sigma W(t)}
\qquad\qquad
\mathbb{E}[X(t)] = X_0\,e^{\mu t} \tag{5.5 - 5.6}$$

---

### §4.1 · $u(x)$ for the toy problem &nbsp;&nbsp;

With $f(x)=\mu \cdot x$ and $g(x)=\sigma \cdot x$, the boundary value problem $\mathcal{A}u=-1$:

$$\tfrac{1}{2}\sigma^2x^2\,u'' + \mu x\,u' = -1
\quad\text{ for } a<x<b, \qquad u(a)=u(b)=0 \tag{12.3}$$

which has the solution

$$u(x) = \frac{1}{\frac{1}{2}\sigma^2-\mu}
\left(\log\frac{x}{a} - \frac{1-(x/a)^{1-2\mu/\sigma^2}}{1-(b/a)^{1-2\mu/\sigma^2}}\,\log\frac{b}{a}\right) \tag{12.4}$$

Our **reference solution** — obtained both *analytically* (12.4) and *numerically*:

In [ ]:
# Mean Exit Time function u(x)

# Experiment: closed form (12.4) of the derived u(x)
#               and the numerical approximation with MC-EM on a grid of 40 starting values x0
#               for each x0; M=10^5 paths, Δt = 10^-1.

fig = figures.fig_u_with_heatmap(TOY)

---

### §4.2 · MC-EM for the toy problem — with exact path updates &nbsp;&nbsp;

For GBM we know the exact solution (5.5), so in **line 7** of the pseudocode, instead of

$$X_n \;\leftarrow\; X_n + \Delta t\,\mu X_n + \sqrt{\Delta t}\,\xi_n\,\sigma X_n
\qquad\qquad\text{(EM step (8.3))}$$

we use

$$X_n \;\leftarrow\; X_n \exp\!\Big(\big(\mu-\tfrac{1}{2}\sigma^2\big)\Delta t + \sqrt{\Delta t}\,\xi_n\,\sigma\Big)
\qquad\text{(exact update, from (5.5))}$$

The grid values now carry **no discretization error**: error source 2 is gone,
and whatever error remains is **sampling (E1) + missed exits (E3)**.

**Experiment**: the full grid $M \in \{10^{3},10^{4},10^{5}\} \times \Delta t \in \{10^{-1},10^{-2},10^{-3}\}$.

In [ ]:
# CI for varying Δt and M

# Experiment: sample mean and 95% CI on the full M × Δt grid,
#               M ∈ { 10^3, 10^4, 10^5 },  Δt ∈ { 10^-1, 10^-2, 10^-3 }.

res42 = grid_sweep(TOY, get_rng, "grid")
fig = figures.fig_m_dt_grid(TOY, res42)

Reading the grid — the two knobs are **orthogonal**:

1. at fixed $\Delta t$ (one cluster), growing $M$ only **tightens** the interval around the same centre;
2. at $\Delta t = 10^{-3}$ the exact answer lies **well outside** the tightest confidence interval (open markers), and
3. the Monte Carlo method **overestimates** — always from above, exactly as the missed-exit picture predicts.

The CI is **honest about what it measures**: the mean exit time of the discretely observed process
$\{t_i, X(t_i)\}$ — a different, larger number than the one we want.

$$\boxed{\text{The confidence interval measures precision, not accuracy.}}$$

- To improve estimation, **decrease $\Delta t$** — increasing $M$ would only sharpen the wrong target.

---

### §4.2 · MC-EM for the toy problem — empirical error convergence &nbsp;&nbsp;

$$\mathrm{err}_{\Delta t} := \big|\,a_M - T^{\mathrm{mean}}_{\mathrm{exit}}\big| \approx C\,\Delta t^{\,q} \tag{12.5}$$

and fit $\log \mathrm{err}_{\Delta t} = \log C + q \log \Delta t$ by **least squares** in log–log coordinates.


The book finds $q = 0.45$ (residual $0.1171$) — consistent with the widely reported
$O(\Delta t^{1/2})$ behaviour.

**Experiment**: fit (12.5) to the $M=10^5$ row of the grid above — no new simulation.

In [ ]:
# Empirical error convergence rate

# Experiment: least-squares fit of (12.5) on the M = 10^5 row of the
#               M × Δt grid from the previous experiment. 
#               for M = 10^5,  Δt ∈ { 10^-1, 10^-2, 10^-3 }

res43 = res42[-1]             
fig, (q, Cfit, resid) = figures.fig_convergence(TOY, res43)

---

## §5 · Outro &nbsp;&nbsp; **What we did**

1. **Problem**: mean exit time 
   1. $T^{\mathrm{mean}}_{\mathrm{exit}} := \mathbb{E}\left[\inf\{t: X(t)=a \;\text{or}\; X(t)=b\}\right]$.
2. **Method**: MC-EM — and its three error sources: 
   1. *sampling*, 
   2. *discretization*, 
   3. *missed exits*.
3. **Baseline**: an analytical dual of our problem
   1. $\;\mathcal{A}u = -1, \,\, u(a)=u(b)=0$ — a deterministic reference.
4. **Toy problem**:
   1. known closed form (12.4); 
   2. exact updates isolate the missed-exit error;
   3. the CIs are *precise but biased*; 
   4. empirically $\mathrm{err} = O(\Delta t^{1/2})$.

---

## §5 · Outro &nbsp;&nbsp;**Questions to keep pondering**

* A natural question from this point: the **probability that the process reaches $a$ before $b$** ?
  
* The order of convergence $O(\Delta t^{1/2})$ is **very bad** — how can it be improved ?
  * adaptive $\Delta t$ near the boundary
  * after each step, compute the probability that an exit was missed and draw a uniform to decide 
  * random exponential stepsizes

* A **rigorous proof** of the error order $O(\Delta t^{1/2})$ ?
  * *(§12.5: not known in this generality — paths may take arbitrarily long to exit, so finite-time convergence theory does not apply)*

* **Other methods** to compute mean exit times ?
  * random-walk-based methods
  * multilevel Monte Carlo
  * apply a numerical method to the deterministic ODE (12.2)